# Alternative Methods for Regime Detection

Implements and compares these non-HMM approaches:
1. **Gaussian Mixture Model (GMM)** - unsupervised, no temporal dynamics  
2. **K-Means Clustering** - hard assignment on feature space  
3. **Isolation Forest** - anomaly/stress period detection  

---

### References  
1. Nguyen & Nguyen (2015). Risks 3(4) - stock selection via HMM  
2. LSEG API Samples. *Market Regime Detection Using Statistical and ML Approaches*  
3. SSRN Paper 3406068. *Portfolio selection with HMMs*

---

## Review note against standard regime-detection practice

This notebook keeps the baseline side intentionally compact: GMM and K-Means are static clustering models, while Isolation Forest is an anomaly detector for stress periods. These methods are useful comparison points because their assignments are independent across time unless a separate smoothing or transition model is added. That makes them a clean contrast with the HMM, which explicitly estimates transition probabilities.

## 0. Imports & Data

In [ ]:
%pip install -q scikit-learn


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

from utils.data_utils import prepare_ticker_data, time_split
from utils.viz_utils  import plot_price_with_regimes, plot_regime_distributions, _DARK_LAYOUT

print("Imports OK")


In [ ]:
TICKER = "SPY"
df = prepare_ticker_data(TICKER, start="2024-01-01")
df_train, df_test = time_split(df, train_frac=0.80)

# Keep all baselines on the same signal as the HMM: daily log returns.
FEATURE_COLS = ["Returns"]
X_feat_train = df_train[FEATURE_COLS].values
X_feat_test  = df_test[FEATURE_COLS].values
X_1d_train   = df_train["Returns"].values
X_1d_test    = df_test["Returns"].values

scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_feat_train)
X_scaled_test  = scaler.transform(X_feat_test)

assert np.isfinite(X_scaled_train).all(), "Scaled features should be finite"

print(f"Train: {len(df_train)}  Test: {len(df_test)}")
print("Features:", FEATURE_COLS)
print("Training return summary:")
print(df_train["Returns"].describe().round(6))


## 1. Gaussian Mixture Model (GMM)

The GMM models the return distribution as a weighted sum of Gaussians **without** the Markov temporal structure:
$$f(r) = \sum_{j=1}^{K} \pi_j \cdot \mathcal{N}(r;\, \mu_j,\, \sigma_j^2)$$

Fitted via EM (same algorithm as Baum-Welch, but no transition matrix).

**Key difference from HMM:** state transitions are IID i.e. the model has no memory of the previous regime.

In [ ]:
# Fit GMM with K=2,3,4,5 and select via BIC
gmm_rows = []
for k in range(2, 6):
    model = GaussianMixture(n_components=k, covariance_type="full",
                            random_state=42, n_init=5, max_iter=300)
    model.fit(X_1d_train.reshape(-1, 1))
    gmm_rows.append({
        "K": k,
        "BIC": model.bic(X_1d_train.reshape(-1, 1)),
        "AIC": model.aic(X_1d_train.reshape(-1, 1)),
        "model": model,
    })

best_k = min(gmm_rows, key=lambda r: r["BIC"])["K"]
print(f"BIC-optimal K = {best_k}")
for row in gmm_rows:
    print(f"  K={row['K']}  BIC={row['BIC']:.1f}  AIC={row['AIC']:.1f}")


In [ ]:
gmm = GaussianMixture(n_components=3, covariance_type="full",
                      random_state=42, n_init=10, max_iter=300)
gmm.fit(X_1d_train.reshape(-1, 1))

gmm_states = gmm.predict(X_1d_train.reshape(-1, 1))

# Sort by std  (low -> high vol)
gmm_stds  = np.sqrt(gmm.covariances_.ravel())
gmm_order = np.argsort(gmm_stds)
gmm_remap = {gmm_order[k]: k for k in range(3)}
gmm_states = np.array([gmm_remap[s] for s in gmm_states])

gmm_df = df_train.copy()
gmm_df["GMM"] = gmm_states

print("GMM means  :", gmm.means_.ravel()[gmm_order].round(6))
print("GMM stds   :", gmm_stds[gmm_order].round(6))
print("GMM weights:", gmm.weights_[gmm_order].round(4))


In [ ]:
fig = plot_price_with_regimes(gmm_df, "GMM", title="SPY: GMM States")
fig.show()

fig = plot_regime_distributions(gmm_df, "GMM", title="GMM Return Distributions")
fig.show()


## 2. K-Means Clustering

Hard cluster assignment on scaled return space.

K-Means minimises:
$$\sum_{k=1}^K \sum_{x \in C_k} \|x - \mu_k\|^2$$

No emission or transition structure is used; the method is purely geometric.

In [ ]:
# Elbow method + silhouette
inertias, silhouettes = [], []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_scaled_train)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled_train, labels_k))
    print(f"K={k}  inertia={km.inertia_:.1f}  silhouette={silhouettes[-1]:.4f}")

fig = make_subplots(rows=1, cols=2, subplot_titles=("Elbow Method", "Silhouette Score"))
fig.add_trace(go.Scatter(x=list(range(2,7)), y=inertias, mode="lines+markers",
    line=dict(color="rgba(99,200,180,0.9)")), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(2,7)), y=silhouettes, mode="lines+markers",
    line=dict(color="rgba(255,195,55,0.9)")), row=1, col=2)
fig.update_layout(height=380, title="K-Means: Choosing K", **_DARK_LAYOUT)
fig.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
km_states = kmeans.fit_predict(X_scaled_train)

# Sort by average return volatility
km_stds  = [X_1d_train[km_states == k].std() for k in range(3)]
km_order = np.argsort(km_stds)
km_remap = {km_order[k]: k for k in range(3)}
km_states = np.array([km_remap[s] for s in km_states])

km_df = df_train.copy()
km_df["KMeans"] = km_states

fig = plot_price_with_regimes(km_df, "KMeans", title="SPY: K-Means States")
fig.show()

fig = plot_regime_distributions(km_df, "KMeans", title="K-Means Return Distributions")
fig.show()



## 3. Isolation Forest : Stress Period Detection

Isolation Forest assigns **anomaly scores** to each day.
Days with anomaly score < -0.05 are labelled as stress periods in this notebook.

In [ ]:
iso = IsolationForest(n_estimators=200, contamination=0.08, random_state=42)
iso.fit(X_scaled_train)
anomaly_scores = iso.decision_function(X_scaled_train)
stress_days = (anomaly_scores < -0.05).astype(int)   # 1 = stress

df_iso = df_train.copy()
df_iso["Stress"] = stress_days

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Anomaly Score", "SPY Price"),
                    row_heights=[0.4, 0.6])

fig.add_trace(go.Scatter(x=df_train.index, y=anomaly_scores,
    name="Anomaly Score", line=dict(color="rgba(255,195,55,0.8)", width=0.8)), row=1, col=1)
fig.add_hline(y=-0.05, line_dash="dash", line_color="rgba(240,80,80,0.8)", row=1, col=1)

fig.add_trace(go.Scatter(x=df_train.index, y=df_train["Close"],
    name="SPY Close", line=dict(color="rgba(150,180,255,0.9)", width=1.2)), row=2, col=1)

for t in df_iso.index[df_iso["Stress"] == 1]:
    fig.add_vrect(x0=t, x1=t, fillcolor="rgba(240,80,80,0.15)", line_width=0, row=2, col=1)

fig.update_layout(title="Isolation Forest: Stress Days",
                  height=550, **_DARK_LAYOUT)
fig.show()

print(f"Stress days: {stress_days.sum()} ({stress_days.mean()*100:.1f}% of sample)")
stress_rets = X_1d_train[stress_days == 1]
normal_rets = X_1d_train[stress_days == 0]
print(f"Stress mean/vol: {stress_rets.mean()*252:.2%} / {stress_rets.std()*np.sqrt(252):.2%}")
print(f"Normal mean/vol: {normal_rets.mean()*252:.2%} / {normal_rets.std()*np.sqrt(252):.2%}")

## 4. Notes and Limitations

### Advantages & Shortcomings

**GMM**
- Clean probabilistic framework, soft assignments  
- Ignores time ordering, two adjacent days treated as independent  
- Gaussian assumption may not capture fat tails  

**K-Means**
- Simple, fast, interpretable  
- Euclidean distance penalises outliers; sensitive to feature scaling  
- No uncertainty estimate; hard boundaries  

**Isolation Forest**
- Useful for pinpointing stress events  
- Binary output, no multi-state regime structure  

**HMM**
- Principled probabilistic model with temporal dynamics (Markov property)  
- Soft state probabilities; joint sequence likelihood  
- Assumes first-order Markov dependence  
- Gaussian emissions may be mis-specified; can overfit